***EDA Escuelas***

Solo se tendrá en cuenta escuelas iniciales, primaria y secundaria. Se filtrará por `nivel_educativo`.

Se filtrará las escuelas que no tengan nombre del estableciemiento. 

Los campos de tipo string se debe limpiar los caracteres raros como "".

Las coordenadas estan metros se debe pasar a grados decimales para poder graficar. Usar `st_transform`.

Las coordenadas son válidas, estan dentro de CABA.



In [0]:
%sql
--Estructura
DESCRIBE proyecto_final.raw.escuelas_bronze;

In [0]:
%sql
--Nulos

with total_registros as (
  select count(*) as total_registros from proyecto_final.raw.escuelas_bronze
),
nulos as (
  select count(*)-count(id_establecimiento) as id_nulos,
  count(*)-count(nombre_funcional) as nombre_funcional_nulos,
  count(*)-count(nombre_generico) as nombre_generico_nulos,
  count(*)-count(nombre_establecimiento) as nombre_establecimiento_nulos,
  count(*)-count(codigo_cui) as codigo_cui_nulos,
  count(*)-count(codigo_cue) as codigo_cue_nulos,
  count(*)-count(codigo_anexo) as codigo_anexo_nulos,
  count(*)-count(tipo_gestion) as tipo_gestion_nulos,
  count(*)-count(nivel_educativo) as nivel_educativo_nulos,
  count(*)-count(tipo_escuela) as tipo_escuela_nulos,
  count(*)-count(direccion) as direccion_nulos,
  count(*)-count(barrio) as barrio_nulos,
  count(*)-count(comuna) as comuna_nulos,
  count(*)-count(fuente_datos) as fuente_datos_nulos,
  count(*)-count(geometry) as geometry_nulos
  from proyecto_final.raw.escuelas_bronze
)
select 
  total_registros,
  id_nulos,
  nombre_funcional_nulos,
  nombre_generico_nulos,
  nombre_establecimiento_nulos,
  codigo_cui_nulos,
  codigo_cue_nulos,
  codigo_anexo_nulos,
  tipo_gestion_nulos,
  nivel_educativo_nulos,
  tipo_escuela_nulos,
  direccion_nulos,
  barrio_nulos,
  comuna_nulos,
  fuente_datos_nulos,
  geometry_nulos
from total_registros, nulos
    
--nombre_establecimiento = 6
/*nombre_establecimiento_nulos, 6 nulos
  codigo_cui_nulos,
  codigo_cue_nulos,
  codigo_anexo_nulos,
  tipo_gestion_nulos,
  nivel_educativo_nulos,
  tipo_escuela_nulos,*/

-- Hay 6 establecimientos que no tienen nombre establecimiento ni  ni cue, que es un codigo identificador por el ministerio


In [0]:
%sql

select * 
from proyecto_final.raw.escuelas_bronze
where nombre_establecimiento is null

-- Hay 6 establecimientos que no tienen nombre establecimiento ni  ni cue, que es un codigo identificador por el ministerio

In [0]:
%sql
--valores unicos tipo gestion

select  tipo_gestion,
count(*) as cantidad,
round(count(*)*100/sum(count(*)) over(),2) as porcentaje
from proyecto_final.raw.escuelas_bronze
group by tipo_gestion;

-- hay mas estatales que privada

In [0]:
%sql

--valores unicos tipo escuela

select  tipo_escuela,
count(*) as cantidad,
round(count(*)*100/sum(count(*)) over(),2) as porcentaje
from proyecto_final.raw.escuelas_bronze
group by tipo_escuela
order by porcentaje desc

In [0]:
%sql
--valores unicos barrios
select barrio,
count(*) as cantidad,
round(count(*)*100/sum(count(*)) over(),2) as porcentaje
from proyecto_final.raw.escuelas_bronze
group by barrio
order by porcentaje desc

In [0]:
%sql
--valores unicos nombre generico
select nombre_generico,
count(*) as cantidad,
round(count(*)*100/sum(count(*)) over(),2) as porcentaje
from proyecto_final.raw.escuelas_bronze
group by nombre_generico
order by porcentaje

In [0]:
%sql
--valores unicos nivel educativo

-- Hay muchos niveles, pero se puede distinguir primario, secundario e inicial
select  nivel_educativo,
count(*) as cantidad,
round(count(*)*100/sum(count(*)) over(),2) as porcentaje
from proyecto_final.raw.escuelas_bronze
group by nivel_educativo
order by porcentaje desc


In [0]:

%sql

-- Entonces solo se tendrá en cuenta escuelas de nivel primario, secundario y jardín, ya que el resto no tendría tanta concentración de personas como esta
select * from proyecto_final.raw.escuelas_bronze
where nivel_educativo RLIKE '(?i)inicial|primario|secundario'
 AND nivel_educativo NOT RLIKE '(?i)profesional|laboral|talleres|superior';

In [0]:
--valores unicos tipo gestion filtrando

select  tipo_gestion,
count(*) as cantidad,
round(count(*)*100/sum(count(*)) over(),2) as porcentaje
from proyecto_final.raw.escuelas_bronze
where nivel_educativo RLIKE '(?i)inicial|primario|secundario'
 AND nivel_educativo NOT RLIKE '(?i)profesional|laboral|talleres|superior'
-- AND nombre_establecimiento is not null -- la condicion antrerior ya los filtra
group by tipo_gestion;


In [0]:
%sql
--valores unicos barrios y distribucion
select barrio,
count(*) as cantidad,
round(count(*)*100/sum(count(*)) over(),2) as porcentaje
from proyecto_final.raw.escuelas_bronze
where nivel_educativo RLIKE '(?i)inicial|primario|secundario'
 AND nivel_educativo NOT RLIKE '(?i)profesional|laboral|talleres|superior'
group by barrio
order by porcentaje desc

In [0]:
%sql 
--valores unicos comuna y distribucion

select comuna,
count(*) as cantidad,
round(count(*)*100/sum(count(*)) over(),2) as porcentaje
from proyecto_final.raw.escuelas_bronze
where nivel_educativo RLIKE '(?i)inicial|primario|secundario'
 AND nivel_educativo NOT RLIKE '(?i)profesional|laboral|talleres|superior'
 group by comuna
order by porcentaje desc

In [0]:
%sql
select distinct geometry from proyecto_final.raw.escuelas_bronze
-- Cambiar el formato actual POINT (x,y) a latitude y longitude
    

In [0]:
%sql
--Duplicados

select 
nombre_establecimiento,
geometry,
count(*)
from proyecto_final.raw.escuelas_bronze
where nivel_educativo RLIKE '(?i)inicial|primario|secundario'
 AND nivel_educativo NOT RLIKE '(?i)profesional|laboral|talleres|superior'
group by nombre_establecimiento, geometry
having count(*) > 1


In [0]:
%sql

with escuelas as(
  select * from proyecto_final.raw.escuelas_bronze
  where nivel_educativo RLIKE '(?i)inicial|primario|secundario'
 AND nivel_educativo NOT RLIKE '(?i)profesional|laboral|talleres|superior'
),
duplicados as (
  select 
  nombre_establecimiento,
  geometry,
  nivel_educativo,
  count(*) as cantidad
  from escuelas
  group by nombre_establecimiento, geometry, nivel_educativo
  having count(*) > 1
)
select count(*) as cantidad_grupos_duplicados,
sum(cantidad) as total_registros_duplicados,
sum(cantidad -1) as registros_extra
from duplicados

In [0]:
SELECT st_astext(st_transform(st_geomfromtext(geometria_wkt, 9498), 4326)) coordenadas, -- long lat,
st_x(st_transform(st_geomfromtext(geometria_wkt, 9498), 4326)) AS longitud,
st_y(st_transform(st_geomfromtext(geometria_wkt, 9498), 4326)) AS latitud
FROM proyecto_final.raw.escuelas_bronze
limit 20;

In [0]:
%sql
-- Si las coordenadas esta dentro de CABA
with coordenadas as (
  SELECT st_astext(st_transform(st_geomfromtext(geometry, 9498), 4326)) coordenadas, -- long lat,
  st_x(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS longitud,
  st_y(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS latitud
  FROM proyecto_final.raw.escuelas_bronze
)
select * from coordenadas
WHERE latitud > -34.50 OR latitud < -34.70 
   OR longitud > -58.30 OR longitud < -58.55;

In [0]:
-- Filtrar por escuelas primarias, secundarias, iniciales
-- Limpiar los datos que tiene ""

-- Normalizar geometria POINT(x,y) por latitud y longitud

In [0]:
CREATE OR REPLACE VIEW v_escuelas_limpieza AS
SELECT 
    -- 1. Identificadores y códigos (se mantienen sin cambios ni casteos)
    id_establecimiento,
    codigo_cui,
    codigo_cue,
    codigo_anexo,
    comuna,

    -- 2. Limpieza de texto: minúsculas, eliminación de comillas dobles y espacios extras
    UPPER(REPLACE(TRIM(nombre_funcional), '"', '')) AS nombre_funcional,
    UPPER(REPLACE(TRIM(nombre_generico), '"', '')) AS nombre_generico,
    UPPER(REPLACE(TRIM(nombre_establecimiento), '"', '')) AS nombre_establecimiento,
    LOWER(REPLACE(TRIM(tipo_gestion), '"', '')) AS tipo_gestion,
    LOWER(REPLACE(TRIM(nivel_educativo), '"', '')) AS nivel_educativo,
    LOWER(REPLACE(TRIM(tipo_escuela), '"', '')) AS tipo_escuela,
    LOWER(REPLACE(TRIM(dependencia), '"', '')) AS dependencia,
    LOWER(REPLACE(TRIM(direccion), '"', '')) AS direccion,
    LOWER(REPLACE(TRIM(barrio), '"', '')) AS barrio,
    LOWER(REPLACE(TRIM(fuente_datos), '"', '')) AS fuente_datos,

    -- 3. Campos de auditoría y geometría
    geometry,
    fecha_ingesta,
    st_x(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS longitud,
  st_y(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS latitud
FROM proyecto_final.raw.escuelas_bronze
-- 4. Filtro de calidad para el EDA
WHERE nombre_establecimiento IS NOT NULL 
  AND geometry IS NOT NULL AND
  nivel_educativo RLIKE '(?i)inicial|primario|secundario'
 AND nivel_educativo NOT RLIKE '(?i)profesional|laboral|talleres|superior';

In [0]:
SELECT * from v_escuelas_limpieza